In [ ]:
from pathlib import Path
import os, sys
WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/branch_sql_MVP/settings.json').is_file())
sys.path.insert(0, str(WORKSPACE)) if str(WORKSPACE) not in sys.path else None
MVP_ROOT = WORKSPACE / 'src/branch_sql_MVP'
os.chdir(MVP_ROOT)


# Benchmark Text-to-SQL: dev tuning → frozen test comparison

Notebook duy nhất lưu không gian tham số, kết quả chọn tham số trên dev, cấu hình đã đóng băng, manifest và so sánh toàn bộ kịch bản trên test. Workflow được triển khai bằng LangGraph 1.x. Gold SQL không đi vào state/prompt; evaluator chỉ đọc gold sau khi prediction đã đóng băng.

In [1]:
from pathlib import Path
import json
import pandas as pd

from src.branch_sql_MVP.eval.run_text2sql_benchmark import (
    BASE_CONTEXT, CANDIDATE_GRID, DEV_CASES_PER_DATABASE, EVENT_GRID,
    EXECUTION_PARAMETERS, MAX_CONCURRENCY, REPAIR_GRID, RETRIEVAL_GRID,
    SCHEMA_GRID, SEED, SELECTOR_GRID, TOKEN_GRID, BUNDLE_PATH,
)

bundle_path = BUNDLE_PATH.resolve()
bundle = json.loads(bundle_path.read_text(encoding='utf-8'))
print(f'Bundle: {bundle_path}')
print(json.dumps(bundle['protocol'], ensure_ascii=False, indent=2))

Bundle: C:\Users\Khanh\Documents\vi-coze\src\branch_sql_MVP\.runtime\text2sql_benchmark\benchmark_bundle.json
{
  "order": "tune only on dev, freeze parameters, report only on test",
  "dev_total_cases": 1534,
  "dev_tuning_cases": 11,
  "dev_sampling": "stratified 1 case/database, seed=42",
  "test_cases": 62,
  "test_scope": "all cases",
  "evaluation": "sqlite_result_equivalence_v1",
  "r_ves": null,
  "example_corpus": "unavailable: repository has no train split"
}


## Không gian tham số và nguyên tắc chọn

Các nhóm được tune tuần tự trên dev: retrieval → schema linking → token budget → event expansion → contextual selector → repair bound → adaptive candidate bound. Mỗi nhóm chỉ thay đổi biến đang xét; test không được dùng để chọn tham số. Nếu độ phủ bằng nhau, context nhỏ hơn/ít bước hơn/ít token hơn được ưu tiên.

In [2]:
parameter_space = {
    'base_context': BASE_CONTEXT,
    'retrieval_grid': RETRIEVAL_GRID,
    'schema_grid': SCHEMA_GRID,
    'token_grid': TOKEN_GRID,
    'event_grid': EVENT_GRID,
    'selector_grid': SELECTOR_GRID,
    'repair_grid': REPAIR_GRID,
    'candidate_grid': CANDIDATE_GRID,
    'execution': EXECUTION_PARAMETERS,
    'dev_cases_per_database': DEV_CASES_PER_DATABASE,
    'max_concurrency': MAX_CONCURRENCY,
    'seed': SEED,
}
print('Spec đồng bộ với bundle:', parameter_space == bundle['parameter_space'])
print(json.dumps(parameter_space, ensure_ascii=False, indent=2))

Spec đồng bộ với bundle: True
{
  "base_context": {
    "mode": "hybrid",
    "candidate_k": 8,
    "docs_top_k": 3,
    "semantic_weight": 0.5,
    "keyword_weight": 0.5,
    "rrf_k": 40,
    "rerank_top_k": 4,
    "rerank_enabled": false,
    "rerank_max_length": 128,
    "rerank_batch_size": 8,
    "table_k": 5,
    "column_k": 12,
    "schema_min_score": 0.12,
    "value_k": 8,
    "value_fuzzy_threshold": 0.84,
    "token_budget": 5000,
    "event_top_k": 8,
    "event_hops": 1,
    "event_node_budget": 24,
    "event_token_budget": 1800,
    "selector_max_items": 24,
    "example_k": 0
  },
  "retrieval_grid": [
    {
      "name": "hybrid_rrf_dense_0_3",
      "semantic_weight": 0.3,
      "keyword_weight": 0.7
    },
    {
      "name": "hybrid_rrf_balanced",
      "semantic_weight": 0.5,
      "keyword_weight": 0.5
    },
    {
      "name": "hybrid_rrf_dense_0_7",
      "semantic_weight": 0.7,
      "keyword_weight": 0.3
    },
    {
      "name": "hybrid_crossencoder_128",
 

## Dev tuning — context/retrieval thuần cục bộ

In [3]:
context_rows = []
for stage in ('retrieval', 'schema', 'token_budget', 'events'):
    for row in bundle['dev_tuning'].get(stage, []):
        context_rows.append({
            'stage': stage, 'name': row['name'], 'cases': row['cases'],
            'schema_complete_rate': row.get('schema_complete_rate'),
            'schema_table_recall': row.get('schema_table_recall'),
            'event_gold_complete_rate': row.get('event_gold_complete_rate'),
            'event_gold_table_recall': row.get('event_gold_table_recall'),
            'mean_context_tokens': row.get('mean_context_tokens'),
            'mean_event_items': row.get('mean_event_items'),
            'elapsed_seconds': row.get('elapsed_seconds'),
        })
pd.DataFrame(context_rows)

,stage,name,cases,schema_complete_rate,schema_table_recall,event_gold_complete_rate,event_gold_table_recall,mean_context_tokens,mean_event_items,elapsed_seconds
0,retrieval,hybrid_rrf_dense_0_3,11,1.0,1.0,NaN,NaN,4370.000000,NaN,7.819141
1,retrieval,hybrid_rrf_balanced,11,1.0,1.0,NaN,NaN,4377.454545,NaN,7.832918
2,retrieval,hybrid_rrf_dense_0_7,11,1.0,1.0,NaN,NaN,4180.545455,NaN,9.650837
3,retrieval,hybrid_crossencoder_128,11,1.0,1.0,NaN,NaN,4140.272727,NaN,25.420610
4,schema,schema_compact,11,1.0,1.0,NaN,NaN,4011.454545,NaN,9.615937
5,schema,schema_balanced,11,1.0,1.0,NaN,NaN,4140.272727,NaN,10.078082
6,schema,schema_wide,11,1.0,1.0,NaN,NaN,4060.636364,NaN,11.481997
7,token_budget,context_3k,11,1.0,1.0,NaN,NaN,2191.636364,NaN,10.272329
8,token_budget,context_5k,11,1.0,1.0,NaN,NaN,4011.454545,NaN,9.652575
9,token_budget,context_7k,11,1.0,1.0,NaN,NaN,4375.454545,NaN,10.367482


In [4]:
print('Dev stable IDs dùng để tune:')
print('\n'.join(bundle['dev_stable_ids']))
print('\nCấu hình context đã khóa:')
print(json.dumps(bundle['selected'].get('frozen_context'), ensure_ascii=False, indent=2))

Dev stable IDs dùng để tune:
vi:dev:california_schools:5
vi:dev:card_games:523
vi:dev:codebase_community:644
vi:dev:debit_card_specializing:1500
vi:dev:european_football_2:1036
vi:dev:financial:143
vi:dev:formula_1:942
vi:dev:student_club:1387
vi:dev:superhero:743
vi:dev:thrombosis_prediction:1289
vi:dev:toxicology:234

Cấu hình context đã khóa:
{
  "mode": "hybrid",
  "candidate_k": 8,
  "docs_top_k": 3,
  "semantic_weight": 0.5,
  "keyword_weight": 0.5,
  "rrf_k": 40,
  "rerank_top_k": 4,
  "rerank_enabled": true,
  "rerank_max_length": 128,
  "rerank_batch_size": 8,
  "table_k": 3,
  "column_k": 8,
  "schema_min_score": 0.12,
  "value_k": 8,
  "value_fuzzy_threshold": 0.84,
  "token_budget": 3000,
  "event_top_k": 4,
  "event_hops": 1,
  "event_node_budget": 24,
  "event_token_budget": 1800,
  "selector_max_items": 12,
  "example_k": 0,
  "contextual_selector": true
}


## Dev tuning — các nút dùng LLM

Bảng này chứa selector size, số vòng repair và số candidate tối đa. Nếu trống, phần benchmark có egress tới OpenAI chưa được cấp quyền/chưa chạy xong.

In [5]:
llm_rows = []
for stage in ('selector', 'repair', 'candidates'):
    for row in bundle['dev_tuning'].get(stage, []):
        metrics = row['metrics']
        llm_rows.append({
            'stage': stage, 'value': row['value'], 'run_id': row['run_id'],
            'execution_accuracy': metrics['execution_accuracy'],
            'invalid_sql_rate': metrics['invalid_sql_rate'],
            'mean_repairs': metrics['mean_repairs'],
            'mean_candidates': metrics['mean_candidates'],
            'input_tokens_all_calls': metrics['input_tokens_all_calls'],
            'output_tokens_all_calls': metrics['output_tokens_all_calls'],
        })
pd.DataFrame(llm_rows)

,stage,value,run_id,execution_accuracy,invalid_sql_rate,mean_repairs,mean_candidates,input_tokens_all_calls,output_tokens_all_calls
0,selector,12,dev-selector-12-799fb03b49,0.272727,0.090909,0.000000,1.000000,60511,3774
1,selector,24,dev-selector-24-1c48158140,0.272727,0.090909,0.000000,1.000000,60616,3924
2,repair,0,dev-repair-0-b02364903e,0.272727,0.090909,0.000000,1.000000,61218,3860
3,repair,1,dev-repair-1-120cdd2fac,0.272727,0.090909,0.181818,1.181818,63260,4145
4,repair,2,dev-repair-2-f8325bccb9,0.272727,0.090909,0.272727,1.272727,63312,4277
5,candidates,1,dev-adaptive-candidates-1-702fc6dcb8,0.363636,0.090909,0.000000,1.000000,41113,2718
6,candidates,2,dev-adaptive-candidates-2-b87fce0d1e,0.545455,0.090909,0.000000,1.636364,67847,4892
7,candidates,3,dev-adaptive-candidates-3-e8fe9ca7f0,0.363636,0.090909,0.000000,1.818182,68284,5256


## Test — toàn bộ kịch bản và ablation

Các dòng test dùng cùng model/prompt và context parameters đã khóa. B4+event seeds, B4+event expansion, B5 selector, B6 với context relational/hybrid và G1 adaptive được tách thành các run độc lập.

In [6]:
test_rows = []
for row in bundle['test_scenarios']:
    metrics = row['metrics']
    manifest = row['manifest']
    test_rows.append({
        'label': row['label'], 'scenario': row['scenario'], 'run_id': row['run_id'],
        'cases': metrics['cases'], 'completed_cases': metrics['completed_cases'],
        'execution_accuracy': metrics['execution_accuracy'],
        'exact_sql_match': metrics['exact_sql_match'],
        'invalid_sql_rate': metrics['invalid_sql_rate'],
        'empty_result_rate': metrics['empty_result_rate'],
        'run_error_rate': metrics['run_error_rate'],
        'mean_generation_latency_seconds': metrics['mean_generation_latency_seconds'],
        'mean_repairs': metrics['mean_repairs'], 'mean_candidates': metrics['mean_candidates'],
        'input_tokens_all_calls': metrics['input_tokens_all_calls'],
        'output_tokens_all_calls': metrics['output_tokens_all_calls'],
        'r_ves': metrics['r_ves'], 'model': manifest['model'],
    })
comparison = pd.DataFrame(test_rows)
comparison.sort_values('execution_accuracy', ascending=False) if not comparison.empty else comparison

,label,scenario,run_id,cases,completed_cases,execution_accuracy,exact_sql_match,invalid_sql_rate,empty_result_rate,run_error_rate,mean_generation_latency_seconds,mean_repairs,mean_candidates,input_tokens_all_calls,output_tokens_all_calls,r_ves,model
0,P1-full-one-shot,P1,test-P1-full-one-shot-3e04e0da09,62,62,0.532258,0.0,0.000000,0.064516,0.0,2.617327,0.0,1.000000,216880,11000,None,gpt-5.4
7,G1-adaptive,G1,test-G1-adaptive-a7be607f9f,62,62,0.500000,0.0,0.000000,0.032258,0.0,2.679337,0.0,1.725806,387202,31904,None,gpt-5.4
2,B4-plus-event-seeds,B5,test-B4-plus-event-seeds-e487026e5f,62,62,0.467742,0.0,0.000000,0.064516,0.0,2.765220,0.0,1.000000,236071,11191,None,gpt-5.4
3,B4-plus-event-expansion,B5,test-B4-plus-event-expansion-960119ab78,62,62,0.451613,0.0,0.000000,0.096774,0.0,2.560548,0.0,1.000000,242019,11298,None,gpt-5.4
1,B4-hybrid-fixed,B4,test-B4-hybrid-fixed-26dcb2ba14,62,62,0.435484,0.0,0.016129,0.080645,0.0,2.784551,0.0,1.000000,205581,11575,None,gpt-5.4
6,B6-repair-hybrid,B6,test-B6-repair-hybrid-78f52fab3e,62,62,0.435484,0.0,0.016129,0.080645,0.0,2.570034,0.0,1.000000,205581,11467,None,gpt-5.4
4,B5-relational-selector,B5,test-B5-relational-selector-864b68c657,62,62,0.419355,0.0,0.000000,0.129032,0.0,2.624592,0.0,1.000000,343694,24071,None,gpt-5.4
5,B6-repair-relational,B6,test-B6-repair-relational-4ae1d6e1d3,62,62,0.403226,0.0,0.000000,0.145161,0.0,2.715217,0.0,1.000000,343910,23924,None,gpt-5.4


In [7]:
print('Artifact fingerprints:')
print(json.dumps({
    'dataset': bundle['artifacts']['dataset_fingerprint'],
    'index': bundle['artifacts']['index_fingerprint'],
    'example_corpus': bundle['artifacts']['example_corpus'],
    'chunk_contract': bundle['artifacts']['chunk_contract'],
}, ensure_ascii=False, indent=2))
print('\nRun manifests:')
print(json.dumps([row['manifest'] for row in bundle['test_scenarios']], ensure_ascii=False, indent=2))

Artifact fingerprints:
{
  "dataset": "88e1f145e36f3dca0e6ae546c03642289f46e5080ab067578920ad91e147c5b2",
  "index": "82673bb6660a408ef13e2e7266cf4ec121147cc76d5f905fffd59c36e66cb25d",
  "example_corpus": "unavailable: repository has no train split",
  "chunk_contract": {
    "artifact_version": 2,
    "settings": {
      "unit": "token",
      "tokenizer_model": null,
      "heading_level": 2,
      "child_min": 40,
      "child_max": 512,
      "child_overlap": 32,
      "parent_max": 3000,
      "table_rows": 10,
      "table_overlap_rows": 1,
      "repeat_table_header": true,
      "separators": [
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
      ],
      "on_overflow": "split",
      "on_underflow": "merge",
      "breadcrumb": true,
      "business_semantic_chunks": true,
      "gold_sample_parent_child": true,
      "gold_query_marker": "-- Query:"
    }
  }
}

Run manifests:
[
  {
    "run_id": "test-P1-full-one-shot-3e04e0da09",
    "scenario": "P1",
 

## Phân rã theo database, độ khó và execution status

Các bảng dưới đây được tính lại từ raw per-case JSONL của đúng run ID trong manifest; denominator gồm toàn bộ case của nhóm.

In [8]:
raw_rows = []
for scenario_row in bundle['test_scenarios']:
    cases_path = Path(bundle['artifacts']['runtime_root']) / 'runs' / scenario_row['run_id'] / 'cases.jsonl'
    for line in cases_path.read_text(encoding='utf-8').splitlines():
        case = json.loads(line)
        raw_rows.append({
            'label': scenario_row['label'], 'db_id': case['db_id'],
            'difficulty': case['difficulty'],
            'execution_correct': float(bool(case.get('execution_correct'))),
            'prediction_status': case.get('prediction_status', 'run_error'),
            'repair_count': case.get('repair_count', 0),
            'candidate_count': len(case.get('candidates', [])),
            'input_tokens': case.get('total_input_tokens', 0),
            'output_tokens': case.get('total_output_tokens', 0),
        })
raw = pd.DataFrame(raw_rows)
by_database = raw.groupby(['label', 'db_id'], as_index=False).agg(
    cases=('execution_correct', 'size'), execution_accuracy=('execution_correct', 'mean'),
    mean_repairs=('repair_count', 'mean'), mean_candidates=('candidate_count', 'mean'),
    input_tokens=('input_tokens', 'sum'), output_tokens=('output_tokens', 'sum'),
)
by_difficulty = raw.groupby(['label', 'difficulty'], as_index=False).agg(
    cases=('execution_correct', 'size'), execution_accuracy=('execution_correct', 'mean'),
)
status_counts = raw.groupby(['label', 'prediction_status']).size().rename('cases').reset_index()
display(by_database)
display(by_difficulty)
display(status_counts)

,label,db_id,cases,execution_accuracy,mean_repairs,mean_candidates,input_tokens,output_tokens
0,B4-hybrid-fixed,debit_card_specializing,30,0.500000,0.0,1.000000,96436,5477
1,B4-hybrid-fixed,financial,32,0.375000,0.0,1.000000,109145,6098
2,B4-plus-event-expansion,debit_card_specializing,30,0.566667,0.0,1.000000,117052,5435
3,B4-plus-event-expansion,financial,32,0.343750,0.0,1.000000,124967,5863
4,B4-plus-event-seeds,debit_card_specializing,30,0.533333,0.0,1.000000,115198,5341
5,B4-plus-event-seeds,financial,32,0.406250,0.0,1.000000,120873,5850
6,B5-relational-selector,debit_card_specializing,30,0.533333,0.0,1.000000,159958,11472
7,B5-relational-selector,financial,32,0.312500,0.0,1.000000,183736,12599
8,B6-repair-hybrid,debit_card_specializing,30,0.500000,0.0,1.000000,96436,5408
9,B6-repair-hybrid,financial,32,0.375000,0.0,1.000000,109145,6059


,label,difficulty,cases,execution_accuracy
0,B4-hybrid-fixed,challenging,11,0.181818
1,B4-hybrid-fixed,moderate,34,0.500000
2,B4-hybrid-fixed,simple,17,0.470588
3,B4-plus-event-expansion,challenging,11,0.090909
4,B4-plus-event-expansion,moderate,34,0.470588
5,B4-plus-event-expansion,simple,17,0.647059
6,B4-plus-event-seeds,challenging,11,0.181818
7,B4-plus-event-seeds,moderate,34,0.470588
8,B4-plus-event-seeds,simple,17,0.647059
9,B5-relational-selector,challenging,11,0.181818


,label,prediction_status,cases
0,B4-hybrid-fixed,empty_result,5
1,B4-hybrid-fixed,missing_object,1
2,B4-hybrid-fixed,success,56
3,B4-plus-event-expansion,empty_result,6
4,B4-plus-event-expansion,success,56
5,B4-plus-event-seeds,empty_result,4
6,B4-plus-event-seeds,success,58
7,B5-relational-selector,empty_result,8
8,B5-relational-selector,success,54
9,B6-repair-hybrid,empty_result,5


## Ghi chú diễn giải

- Repository không có train split, vì vậy similar-example retrieval bị khóa ở trạng thái unavailable; dev gold không bao giờ được index làm example.
- `execution_accuracy` là SQLite result equivalence có xử lý thứ tự khi gold dùng `ORDER BY`; đây không phải official BIRD R-VES. R-VES để `None`, không suy diễn.
- Raw per-case prediction, candidate, observation, trajectory và token usage nằm trong `.runtime/text2sql_benchmark/runs/<run_id>/cases.jsonl`; notebook giữ toàn bộ tham số và bảng so sánh tổng hợp.
- Chạy lại có resume theo stable ID và thread ID duy nhất cho từng LangGraph case.